# APN + Residual RL Path Planning Interceptor

Trains 3 separate Stable-Baselines3 (PPO) models (`QUADCOPTER`, `FIXED_WING`, `MISSILE`) learning world-frame residual acceleration commands on top of 3D Augmented Proportional Navigation (APN) against an unconstrained generic evasive target executing all 15 `ManeuverType` laws.

### Architecture Summary
- **Option A (3 Separate Platform Models)**: Dedicated `PPO` policies for `QUADCOPTER` (omnidirectional, $v_{\min}=0$ hover-capable), `FIXED_WING` (non-holonomic, stall speed $v_{\min}=20\text{ m/s}$), and `MISSILE` (high-speed non-holonomic, $v_{\min}=180\text{ m/s}$).
- **Generic Unconstrained Target**: Executes all 15 closed-loop evasive maneuvers (`NONE` + 14 evasive laws) scaled to the engagement speed regime without platform stall/slew clipping.
- **Standardized World Frame**: 3D NED inertial frame ($[x, y, z]^T$, altitude $h = -z$) with world-frame acceleration actions $[a_x, a_y, a_z] \in [-1, 1]^3$ projected onto each interceptor's kinematic envelope.

In [ ]:
!pip install -q gymnasium stable-baselines3[extra] torch numpy matplotlib tqdm rich

In [ ]:
import os
import math
from dataclasses import dataclass
from enum import Enum
from collections import deque
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import torch as th
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor

G_ACCEL = 9.81


class PlatformType(Enum):
    QUADCOPTER = "quadcopter"
    FIXED_WING = "fixed_wing"
    MISSILE = "missile"


class ManeuverType(Enum):
    NONE = "none"
    CONSTANT_G = "constant_g"
    SINUSOIDAL_WEAVE = "sinusoidal_weave"
    BARREL_ROLL = "barrel_roll"
    TERMINAL_BREAK = "terminal_break"
    ENERGY_DEPLETION = "energy_depletion"
    SPLIT_S = "split_s"
    SQUARE_BANG_BANG = "square_bang_bang"
    SCISSOR = "scissor"
    NOTCHING = "notching"
    TURN_INTO_MISSILE = "turn_into_missile"
    CRANK = "crank"
    SPEED_CHANGE = "speed_change"
    MINIMUM_RANGE_TURN = "minimum_range_turn"
    DIVE_DENSE_AIR = "dive_dense_air"
    SPIRAL_CORKSCREW = "spiral_corkscrew"


@dataclass
class InterceptorLimits:
    platform: PlatformType
    v_min: float
    v_max: float
    a_t_max: float
    a_t_min: float
    a_n_max: float
    a_n_down_max: float
    tau_accel: float
    max_jerk: float
    can_hover: bool
    omnidirectional: bool
    kill_radius: float
    r_init_min: float
    r_init_max: float
    alt_init: float
    v_target_nominal: float
    a_target_max: float
    dt: float
    max_time: float

    @classmethod
    def from_preset(cls, platform: PlatformType) -> "InterceptorLimits":
        if platform == PlatformType.QUADCOPTER:
            return cls(
                platform=platform,
                v_min=0.0,
                v_max=40.0,
                a_t_max=2.5 * G_ACCEL,
                a_t_min=-2.5 * G_ACCEL,
                a_n_max=4.0 * G_ACCEL,
                a_n_down_max=1.0 * G_ACCEL,
                tau_accel=0.06,
                max_jerk=250.0,
                can_hover=True,
                omnidirectional=True,
                kill_radius=3.0,
                r_init_min=150.0,
                r_init_max=450.0,
                alt_init=150.0,
                v_target_nominal=18.0,
                a_target_max=2.5 * G_ACCEL,
                dt=0.02,
                max_time=18.0,
            )
        elif platform == PlatformType.FIXED_WING:
            return cls(
                platform=platform,
                v_min=20.0,
                v_max=240.0,
                a_t_max=2.0 * G_ACCEL,
                a_t_min=-1.5 * G_ACCEL,
                a_n_max=9.0 * G_ACCEL,
                a_n_down_max=4.0 * G_ACCEL,
                tau_accel=0.20,
                max_jerk=120.0,
                can_hover=False,
                omnidirectional=False,
                kill_radius=12.0,
                r_init_min=1200.0,
                r_init_max=3000.0,
                alt_init=2000.0,
                v_target_nominal=110.0,
                a_target_max=5.0 * G_ACCEL,
                dt=0.02,
                max_time=22.0,
            )
        elif platform == PlatformType.MISSILE:
            return cls(
                platform=platform,
                v_min=180.0,
                v_max=1200.0,
                a_t_max=15.0 * G_ACCEL,
                a_t_min=-4.0 * G_ACCEL,
                a_n_max=35.0 * G_ACCEL,
                a_n_down_max=35.0 * G_ACCEL,
                tau_accel=0.10,
                max_jerk=800.0,
                can_hover=False,
                omnidirectional=False,
                kill_radius=25.0,
                r_init_min=3500.0,
                r_init_max=7500.0,
                alt_init=4500.0,
                v_target_nominal=420.0,
                a_target_max=12.0 * G_ACCEL,
                dt=0.02,
                max_time=20.0,
            )
        raise ValueError(f"Unknown platform: {platform}")


In [ ]:
@dataclass
class EngagementKinematics:
    r_vec: np.ndarray
    r: float
    v_rel: np.ndarray
    v_cl: float
    los_vec: np.ndarray
    los_rate_vec: np.ndarray
    t_go: float
    theta_aspect: float
    delta_lead: float
    zem_perp: np.ndarray
    zem_norm: float


def compute_engagement_kinematics(
    p_T: np.ndarray, v_T: np.ndarray, a_T: np.ndarray,
    p_M: np.ndarray, v_M: np.ndarray
) -> EngagementKinematics:
    r_vec = p_T - p_M
    r = float(np.linalg.norm(r_vec))
    if r < 1e-6:
        r = 1e-6
    los_vec = r_vec / r

    v_rel = v_T - v_M
    v_cl = float(-np.dot(v_rel, los_vec))
    los_rate_vec = (v_rel + v_cl * los_vec) / r
    t_go = float(np.clip(r / max(v_cl, 1.0), 0.05, 25.0)) if v_cl > 0 else 25.0

    speed_T = float(np.linalg.norm(v_T))
    speed_M = float(np.linalg.norm(v_M))

    theta_aspect = float(np.arccos(np.clip(np.dot(v_T / speed_T, los_vec), -1.0, 1.0))) if speed_T > 1e-3 else 0.0
    delta_lead = float(np.arccos(np.clip(np.dot(v_M / speed_M, los_vec), -1.0, 1.0))) if speed_M > 1e-3 else 0.0

    zem_vec = v_rel * t_go + 0.5 * a_T * (t_go ** 2)
    zem_perp = zem_vec - np.dot(zem_vec, los_vec) * los_vec
    zem_norm = float(np.linalg.norm(zem_perp))

    return EngagementKinematics(
        r_vec=r_vec,
        r=r,
        v_rel=v_rel,
        v_cl=v_cl,
        los_vec=los_vec,
        los_rate_vec=los_rate_vec,
        t_go=t_go,
        theta_aspect=theta_aspect,
        delta_lead=delta_lead,
        zem_perp=zem_perp,
        zem_norm=zem_norm,
    )


def compute_3d_apn_command(kin: EngagementKinematics, a_T: np.ndarray, N_nav: float = 4.0) -> np.ndarray:
    """Exact 3D ZEM-equivalent Augmented Proportional Navigation command in world frame."""
    v_rel_perp = kin.v_rel - np.dot(kin.v_rel, kin.los_vec) * kin.los_vec
    a_T_perp = a_T - np.dot(a_T, kin.los_vec) * kin.los_vec
    a_pn = N_nav * v_rel_perp / max(kin.t_go, 0.05)
    a_aug = 0.5 * N_nav * a_T_perp
    return a_pn + a_aug


class GenericUnconstrainedTarget:
    """
    Generic unconstrained 3D point-mass target executing all 15 ManeuverType laws
    in world frame (NED) without platform stall, roll-slew, or actuator lag restrictions.
    """

    def __init__(
        self,
        p0: np.ndarray,
        v0: np.ndarray,
        maneuver: ManeuverType,
        v_nominal: float,
        a_max: float,
        rng: np.random.Generator,
    ):
        self.p_W = np.array(p0, dtype=float)
        self.v_W = np.array(v0, dtype=float)
        self.a_W = np.zeros(3, dtype=float)
        self.maneuver = maneuver
        self.v_nominal = float(v_nominal)
        self.a_max = float(a_max)
        self.rng = rng

        self.psi_ref = float(np.arctan2(self.v_W[1], self.v_W[0]))
        self.scissor_sign = 1.0 if rng.random() > 0.5 else -1.0
        self.split_s_completed = False
        self.phi_phase = float(rng.uniform(0.0, 2.0 * np.pi))
        self.omega_base = float(rng.uniform(1.0, 2.2))

    def _unit_tangent_and_normals(self) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        speed = float(np.linalg.norm(self.v_W))
        if speed < 1e-3:
            t_hat = np.array([np.cos(self.psi_ref), np.sin(self.psi_ref), 0.0])
        else:
            t_hat = self.v_W / speed

        horiz_norm = np.hypot(t_hat[0], t_hat[1])
        if horiz_norm < 1e-3:
            n_h = np.array([0.0, 1.0, 0.0])
        else:
            n_h = np.array([-t_hat[1] / horiz_norm, t_hat[0] / horiz_norm, 0.0])

        n_v = -np.cross(t_hat, n_h)
        n_v /= max(np.linalg.norm(n_v), 1e-6)
        return t_hat, n_h, n_v

    def compute_maneuver_command(self, t: float, kin: EngagementKinematics, v_M: np.ndarray) -> np.ndarray:
        t_hat, n_h, n_v = self._unit_tangent_and_normals()
        speed_T = float(np.linalg.norm(self.v_W))
        m = self.maneuver

        if m == ManeuverType.NONE:
            return np.zeros(3)

        elif m == ManeuverType.CONSTANT_G:
            return self.a_max * n_h if t >= 0.3 else np.zeros(3)

        elif m == ManeuverType.SINUSOIDAL_WEAVE:
            return self.a_max * np.sin(self.omega_base * t + self.phi_phase) * n_h

        elif m == ManeuverType.BARREL_ROLL:
            phase = self.omega_base * t + self.phi_phase
            return self.a_max * (np.cos(phase) * n_h + np.sin(phase) * n_v)

        elif m == ManeuverType.TERMINAL_BREAK:
            if kin.t_go <= 2.5:
                cross_1 = np.cross(t_hat, kin.los_vec)
                u_perp = np.cross(cross_1, t_hat)
                norm_u = np.linalg.norm(u_perp)
                u_perp = u_perp / norm_u if norm_u > 1e-6 else n_h
                return self.a_max * u_perp
            return np.zeros(3)

        elif m == ManeuverType.ENERGY_DEPLETION:
            los_perp = kin.los_vec - np.dot(kin.los_vec, t_hat) * t_hat
            norm_lp = np.linalg.norm(los_perp)
            steer_dir = los_perp / norm_lp if norm_lp > 1e-6 else n_h
            a_turn = 0.7 * self.a_max * steer_dir
            a_weave = 0.4 * self.a_max * np.sin(self.omega_base * t) * n_h
            return 0.3 * self.a_max * t_hat + a_turn + a_weave

        elif m == ManeuverType.SPLIT_S:
            current_psi = float(np.arctan2(self.v_W[1], self.v_W[0]))
            delta_psi = abs(np.arctan2(np.sin(current_psi - self.psi_ref), np.cos(current_psi - self.psi_ref)))
            if delta_psi > 2.8 and abs(self.v_W[2]) < 0.2 * max(speed_T, 1.0):
                self.split_s_completed = True
            if self.split_s_completed:
                return 0.2 * self.a_max * t_hat
            return -self.a_max * n_v + 0.2 * self.a_max * t_hat

        elif m == ManeuverType.SQUARE_BANG_BANG:
            sign_val = 1.0 if np.sin(self.omega_base * t + self.phi_phase) >= 0.0 else -1.0
            return self.a_max * sign_val * n_h

        elif m == ManeuverType.SCISSOR:
            psi_max = np.deg2rad(65.0)
            current_psi = float(np.arctan2(self.v_W[1], self.v_W[0]))
            delta_psi = float(np.arctan2(np.sin(current_psi - self.psi_ref), np.cos(current_psi - self.psi_ref)))
            if delta_psi >= psi_max:
                self.scissor_sign = -1.0
            elif delta_psi <= -psi_max:
                self.scissor_sign = 1.0
            return -0.3 * self.a_max * t_hat + self.scissor_sign * self.a_max * n_h

        elif m == ManeuverType.NOTCHING:
            radial_vel = float(np.dot(self.v_W, kin.los_vec))
            los_perp = kin.los_vec - np.dot(kin.los_vec, t_hat) * t_hat
            norm_lp = np.linalg.norm(los_perp)
            n_lambda = los_perp / norm_lp if norm_lp > 1e-6 else n_h
            a_centripetal = (speed_T ** 2) / max(kin.r, 50.0)
            a_mag = np.clip(-4.0 * radial_vel + a_centripetal, -self.a_max, self.a_max)
            return a_mag * n_lambda

        elif m == ManeuverType.TURN_INTO_MISSILE:
            threat_dir = -kin.los_vec
            perp_vec = threat_dir - np.dot(threat_dir, t_hat) * t_hat
            norm_p = np.linalg.norm(perp_vec)
            turn_dir = perp_vec / norm_p if norm_p > 1e-6 else n_h
            return self.a_max * turn_dir + 0.2 * self.a_max * t_hat

        elif m == ManeuverType.CRANK:
            theta_gimbal = np.deg2rad(60.0)
            los_azimuth = float(np.arctan2(-kin.los_vec[1], -kin.los_vec[0]))
            side = 1.0 if np.sin(0.35 * t + self.phi_phase) >= 0.0 else -1.0
            psi_des = los_azimuth + side * theta_gimbal
            current_psi = float(np.arctan2(self.v_W[1], self.v_W[0]))
            psi_err = float(np.arctan2(np.sin(psi_des - current_psi), np.cos(psi_des - current_psi)))
            a_lat = np.clip(3.0 * max(speed_T, 5.0) * psi_err, -self.a_max, self.a_max)
            return a_lat * n_h

        elif m == ManeuverType.SPEED_CHANGE:
            sign_val = 1.0 if np.sin(1.2 * t + self.phi_phase) >= 0.0 else -1.0
            return 0.8 * self.a_max * sign_val * t_hat

        elif m == ManeuverType.MINIMUM_RANGE_TURN:
            if kin.t_go <= 4.0:
                return -0.4 * self.a_max * t_hat + self.a_max * n_h
            return np.zeros(3)

        elif m == ManeuverType.DIVE_DENSE_AIR:
            gamma_dive = np.deg2rad(-30.0) if -self.p_W[2] > 100.0 else 0.0
            horiz = np.hypot(self.v_W[0], self.v_W[1])
            current_gamma = float(np.arctan2(-self.v_W[2], max(horiz, 1e-6)))
            a_pitch = np.clip(3.0 * max(speed_T, 5.0) * (gamma_dive - current_gamma), -self.a_max, self.a_max)
            return 0.3 * self.a_max * t_hat + a_pitch * n_v

        elif m == ManeuverType.SPIRAL_CORKSCREW:
            phi_s = self.omega_base * t + 0.25 * (t ** 2) + self.phi_phase
            return 0.2 * self.a_max * t_hat + self.a_max * (np.cos(phi_s) * n_h + np.sin(phi_s) * n_v)

        return np.zeros(3)

    def step(self, dt: float, t: float, p_M: np.ndarray, v_M: np.ndarray) -> EngagementKinematics:
        kin = compute_engagement_kinematics(self.p_W, self.v_W, self.a_W, p_M, v_M)
        a_cmd = self.compute_maneuver_command(t, kin, v_M)

        norm_a = float(np.linalg.norm(a_cmd))
        if norm_a > self.a_max:
            a_cmd = a_cmd * (self.a_max / norm_a)

        self.a_W = a_cmd
        self.v_W = self.v_W + self.a_W * dt

        speed = float(np.linalg.norm(self.v_W))
        v_hi = 1.6 * self.v_nominal
        v_lo = 0.15 * self.v_nominal
        if speed > v_hi:
            self.v_W = (self.v_W / speed) * v_hi
        elif speed < v_lo and speed > 1e-6:
            self.v_W = (self.v_W / speed) * v_lo

        self.p_W = self.p_W + self.v_W * dt
        if self.p_W[2] > -30.0:
            self.p_W[2] = -30.0
            if self.v_W[2] > 0.0:
                self.v_W[2] = -0.5 * self.v_W[2]

        return kin


In [ ]:
class InterceptorPathPlanningEnv(gym.Env):
    """
    3D World-Frame (NED) Path Planning Gymnasium Environment.
    Supports QUADCOPTER, FIXED_WING, and MISSILE interceptors against all 15 ManeuverTypes.
    Mode: use_apn = True
    """
    metadata = {"render_modes": []}

    def __init__(
        self,
        platform: PlatformType = PlatformType.QUADCOPTER,
        use_apn: bool = True,
        fixed_maneuver: Optional[ManeuverType] = None,
        history_len: int = 4,
        N_nav: float = 4.0,
        seed: Optional[int] = None,
    ):
        super().__init__()
        self.platform = platform
        self.limits = InterceptorLimits.from_preset(platform)
        self.use_apn = use_apn
        self.fixed_maneuver = fixed_maneuver
        self.history_len = history_len
        self.N_nav = N_nav
        self.dt = self.limits.dt
        self.max_steps = int(self.limits.max_time / self.dt)

        self.rng = np.random.default_rng(seed)

        # World-frame 3D acceleration action [-1, 1]^3
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(3,), dtype=np.float32)

        # Single frame = 21 normalized kinematic features stacked over history_len frames
        self.frame_dim = 21
        self.observation_space = spaces.Box(
            low=-10.0, high=10.0, shape=(self.frame_dim * self.history_len,), dtype=np.float32
        )

        self._history = deque(maxlen=self.history_len)
        self.p_M = np.zeros(3)
        self.v_M = np.zeros(3)
        self.a_M = np.zeros(3)
        self.target: Optional[GenericUnconstrainedTarget] = None
        self.step_count = 0
        self.prev_dist = 0.0
        self.prev_action = np.zeros(3)
        self.min_miss_distance = float("inf")
        self.traj_M: List[np.ndarray] = []
        self.traj_T: List[np.ndarray] = []

    def _apply_interceptor_constraints(self, a_cmd_world: np.ndarray) -> np.ndarray:
        """Projects a 3D world-frame acceleration command onto the interceptor's mobility envelope."""
        lim = self.limits
        if lim.omnidirectional:
            norm_a = float(np.linalg.norm(a_cmd_world))
            if norm_a > lim.a_n_max:
                a_cmd_world = a_cmd_world * (lim.a_n_max / norm_a)
            return a_cmd_world

        speed = float(np.linalg.norm(self.v_M))
        t_hat = self.v_M / max(speed, 1e-6)
        a_t = float(np.dot(a_cmd_world, t_hat))
        a_n_vec = a_cmd_world - a_t * t_hat

        a_t = float(np.clip(a_t, lim.a_t_min, lim.a_t_max))
        if speed <= lim.v_min and a_t < 0.0:
            a_t = 0.0
        elif speed >= lim.v_max and a_t > 0.0:
            a_t = 0.0

        a_n_mag = float(np.linalg.norm(a_n_vec))
        if a_n_mag > lim.a_n_max:
            a_n_vec = a_n_vec * (lim.a_n_max / a_n_mag)

        return a_t * t_hat + a_n_vec

    def _build_frame(self, kin: EngagementKinematics, a_apn: np.ndarray) -> np.ndarray:
        lim = self.limits
        r_scale = lim.r_init_max
        v_scale = lim.v_max
        a_scale = lim.a_n_max

        frame = np.concatenate([
            kin.r_vec / r_scale,
            kin.v_rel / v_scale,
            self.v_M / v_scale,
            self.a_M / a_scale,
            self.target.a_W / max(lim.a_target_max, 1.0),
            kin.los_rate_vec,
            [kin.r / r_scale],
            [kin.v_cl / v_scale],
            [kin.t_go / lim.max_time],
        ])
        return np.clip(frame, -10.0, 10.0).astype(np.float32)

    def reset(self, *, seed: Optional[int] = None, options: Optional[dict] = None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)

        lim = self.limits
        self.step_count = 0
        self.p_M = np.array([0.0, 0.0, -lim.alt_init], dtype=float)

        dist = float(self.rng.uniform(lim.r_init_min, lim.r_init_max))
        az = float(self.rng.uniform(-np.deg2rad(35.0), np.deg2rad(35.0)))
        el = float(self.rng.uniform(-np.deg2rad(15.0), np.deg2rad(15.0)))
        dir_to_T = np.array([np.cos(el) * np.cos(az), np.cos(el) * np.sin(az), -np.sin(el)])
        p_T0 = self.p_M + dist * dir_to_T
        p_T0[2] = min(p_T0[2], -50.0)

        heading_T = float(self.rng.uniform(-np.pi, np.pi))
        pitch_T = float(self.rng.uniform(-np.deg2rad(10.0), np.deg2rad(10.0)))
        v_T_mag = float(self.rng.uniform(0.75 * lim.v_target_nominal, 1.15 * lim.v_target_nominal))
        v_T0 = v_T_mag * np.array([
            np.cos(pitch_T) * np.cos(heading_T),
            np.cos(pitch_T) * np.sin(heading_T),
            -np.sin(pitch_T)
        ])

        init_speed = float(self.rng.uniform(max(lim.v_min, 0.45 * lim.v_max), 0.75 * lim.v_max))
        self.v_M = init_speed * dir_to_T
        self.a_M = np.zeros(3, dtype=float)

        maneuver = self.fixed_maneuver if self.fixed_maneuver is not None else self.rng.choice(list(ManeuverType))
        self.target = GenericUnconstrainedTarget(
            p0=p_T0, v0=v_T0, maneuver=maneuver,
            v_nominal=lim.v_target_nominal, a_max=lim.a_target_max, rng=self.rng
        )

        kin = compute_engagement_kinematics(self.target.p_W, self.target.v_W, self.target.a_W, self.p_M, self.v_M)
        a_apn = compute_3d_apn_command(kin, self.target.a_W, self.N_nav) if self.use_apn else np.zeros(3)

        self.prev_dist = kin.r
        self.min_miss_distance = kin.r
        self.prev_action = np.zeros(3, dtype=float)
        self.traj_M = [self.p_M.copy()]
        self.traj_T = [self.target.p_W.copy()]

        self._history.clear()
        first_frame = self._build_frame(kin, a_apn)
        for _ in range(self.history_len):
            self._history.append(first_frame)

        obs = np.concatenate(list(self._history)).astype(np.float32)
        return obs, {"maneuver": maneuver.value, "distance": kin.r}

    def step(self, action: np.ndarray):
        lim = self.limits
        act = np.clip(np.asarray(action, dtype=float).reshape(3), -1.0, 1.0)
        t = self.step_count * self.dt

        kin_pre = self.target.step(self.dt, t, self.p_M, self.v_M)

        if self.use_apn:
            a_apn = compute_3d_apn_command(kin_pre, self.target.a_W, self.N_nav)
            a_rl = act * (0.65 * lim.a_n_max)
            a_raw_world = a_apn + a_rl
        else:
            a_apn = np.zeros(3)
            a_rl = act * lim.a_n_max
            a_raw_world = a_rl

        a_cmd = self._apply_interceptor_constraints(a_raw_world)
        da_dt = (a_cmd - self.a_M) / max(lim.tau_accel, 1e-3)
        jerk_mag = float(np.linalg.norm(da_dt))
        if jerk_mag > lim.max_jerk:
            da_dt = da_dt * (lim.max_jerk / jerk_mag)

        p_M_prev = self.p_M.copy()
        p_T_prev = self.target.p_W - self.target.v_W * self.dt

        self.a_M = self.a_M + da_dt * self.dt
        self.v_M = self.v_M + self.a_M * self.dt

        speed = float(np.linalg.norm(self.v_M))
        if not lim.can_hover and speed < lim.v_min:
            self.v_M = (self.v_M / max(speed, 1e-6)) * lim.v_min
        elif speed > lim.v_max:
            self.v_M = (self.v_M / speed) * lim.v_max

        self.p_M = self.p_M + self.v_M * self.dt
        self.step_count += 1

        self.traj_M.append(self.p_M.copy())
        self.traj_T.append(self.target.p_W.copy())

        dr0 = p_T_prev - p_M_prev
        dv = (self.target.p_W - p_T_prev) - (self.p_M - p_M_prev)
        dv2 = float(np.dot(dv, dv))
        tau_star = float(np.clip(-np.dot(dr0, dv) / max(dv2, 1e-9), 0.0, 1.0)) if dv2 > 1e-9 else 1.0
        substep_dist = float(np.linalg.norm(dr0 + tau_star * dv))

        kin = compute_engagement_kinematics(self.target.p_W, self.target.v_W, self.target.a_W, self.p_M, self.v_M)
        eff_dist = min(kin.r, substep_dist)
        if eff_dist < self.min_miss_distance:
            self.min_miss_distance = eff_dist

        killed = bool(eff_dist <= lim.kill_radius)
        crashed = bool(self.p_M[2] > 0.0)
        diverged = bool(self.step_count > 25 and kin.v_cl < -0.15 * lim.v_max and kin.r > self.min_miss_distance + 40.0)
        timeout = bool(self.step_count >= self.max_steps)

        r_progress = 2.5 * (self.prev_dist - kin.r) / max(lim.v_max * self.dt, 1e-3)
        r_los = -0.15 * float(np.linalg.norm(kin.los_rate_vec))
        r_zem = -0.25 * (kin.zem_norm / max(kin.r, 10.0))
        r_energy = -0.02 * float(np.linalg.norm(a_rl) / lim.a_n_max) ** 2
        r_smooth = -0.02 * float(np.sum((act - self.prev_action) ** 2))
        reward = r_progress + r_los + r_zem + r_energy + r_smooth - 0.01

        if killed:
            time_bonus = 50.0 * (1.0 - self.step_count / self.max_steps)
            reward += 150.0 + time_bonus
        elif crashed:
            reward -= 80.0
        elif diverged:
            reward -= 30.0 * (self.min_miss_distance / lim.r_init_max)

        self.prev_dist = kin.r
        self.prev_action = act.copy()

        self._history.append(self._build_frame(kin, a_apn))
        obs = np.concatenate(list(self._history)).astype(np.float32)

        terminated = killed or crashed or diverged
        truncated = bool(timeout and not terminated)

        info = {
            "killed": killed,
            "crashed": crashed,
            "miss_distance": self.min_miss_distance,
            "maneuver": self.target.maneuver.value,
            "platform": self.platform.value,
            "steps": self.step_count,
        }
        return obs, float(reward), terminated, truncated, info


In [ ]:
class InterceptMetricsCallback(BaseCallback):
    """Logs rolling intercept kill rate and minimum miss distance during SB3 training."""
    def __init__(self, eval_freq: int = 10000, verbose: int = 1):
        super().__init__(verbose)
        self.eval_freq = eval_freq
        self.kills = deque(maxlen=200)
        self.misses = deque(maxlen=200)

    def _on_step(self) -> bool:
        for done, info in zip(self.locals.get("dones", []), self.locals.get("infos", [])):
            if done and "killed" in info:
                self.kills.append(1.0 if info["killed"] else 0.0)
                self.misses.append(info["miss_distance"])
        if self.n_calls % self.eval_freq == 0 and len(self.kills) > 0:
            kr = 100.0 * np.mean(self.kills)
            mm = np.mean(self.misses)
            print(f"  [Step {self.num_timesteps:>7d}] Rolling Kill Rate: {kr:5.1f}% | Mean Miss Distance: {mm:6.2f} m")
        return True


def make_env_fn(platform: PlatformType, rank: int, seed: int = 42):
    def _init():
        env = InterceptorPathPlanningEnv(platform=platform, use_apn=True, seed=seed + rank)
        return Monitor(env)
    return _init


def train_all_platforms(total_timesteps_per_platform: int = 300_000, n_envs: int = 8, save_dir: str = "checkpoints_apn_rl"):
    os.makedirs(save_dir, exist_ok=True)
    trained_models = {}

    for platform in [PlatformType.QUADCOPTER, PlatformType.FIXED_WING, PlatformType.MISSILE]:
        print("=" * 80)
        print(f"Training APN_RL Model for Platform: {platform.value.upper()}")
        print("=" * 80)

        vec_env = DummyVecEnv([make_env_fn(platform, i) for i in range(n_envs)])
        policy_kwargs = dict(
            net_arch=dict(pi=[256, 256, 128], vf=[256, 256, 128]),
            activation_fn=th.nn.SiLU,
        )

        model = PPO(
            "MlpPolicy",
            vec_env,
            learning_rate=3e-4,
            n_steps=1024,
            batch_size=256,
            n_epochs=10,
            gamma=0.995,
            gae_lambda=0.95,
            clip_range=0.2,
            ent_coef=0.005,
            policy_kwargs=policy_kwargs,
            verbose=0,
            device="auto",
            seed=42,
        )

        cb = InterceptMetricsCallback(eval_freq=max(5000, total_timesteps_per_platform // (n_envs * 10)))
        model.learn(total_timesteps=total_timesteps_per_platform, callback=cb, progress_bar=True)

        model_path = os.path.join(save_dir, f"apn_rl_{platform.value}.zip")
        model.save(model_path)
        print(f"Saved checkpoint: {model_path}\n")
        trained_models[platform] = model
        vec_env.close()

    return trained_models


# Adjust `total_timesteps_per_platform` (e.g. 500_000 to 1_500_000 on Colab GPU for full convergence)
TOTAL_TIMESTEPS = 300_000
trained_models = train_all_platforms(total_timesteps_per_platform=TOTAL_TIMESTEPS, n_envs=8)


In [ ]:
def evaluate_matrix(models: Dict[PlatformType, PPO], episodes_per_cell: int = 5):
    """
    Evaluates all 3 trained interceptor models across all 15 ManeuverTypes
    and compares against a Pure APN baseline (zero RL action).
    """
    results = {}
    print(f"{'Platform':<12} | {'Maneuver':<22} | {'APN_RL Kill %':<15} | {'APN_RL Miss (m)':<16} | {'Pure APN Miss (m)':<17}")
    print("-" * 92)

    for platform in [PlatformType.QUADCOPTER, PlatformType.FIXED_WING, PlatformType.MISSILE]:
        model = models[platform]
        results[platform.value] = {}

        for maneuver in ManeuverType:
            rl_kills, rl_misses, apn_misses = [], [], []

            for ep in range(episodes_per_cell):
                env_rl = InterceptorPathPlanningEnv(platform=platform, use_apn=True, fixed_maneuver=maneuver, seed=1000 + ep)
                obs, _ = env_rl.reset()
                done = False
                while not done:
                    action, _ = model.predict(obs, deterministic=True)
                    obs, _, terminated, truncated, info = env_rl.step(action)
                    done = terminated or truncated
                rl_kills.append(1.0 if info["killed"] else 0.0)
                rl_misses.append(info["miss_distance"])

                env_apn = InterceptorPathPlanningEnv(platform=platform, use_apn=True, fixed_maneuver=maneuver, seed=1000 + ep)
                env_apn.reset()
                done_apn = False
                while not done_apn:
                    _, _, term_a, trunc_a, info_apn = env_apn.step(np.zeros(3, dtype=np.float32))
                    done_apn = term_a or trunc_a
                apn_misses.append(info_apn["miss_distance"])

            kr = 100.0 * float(np.mean(rl_kills))
            mm_rl = float(np.mean(rl_misses))
            mm_apn = float(np.mean(apn_misses))
            results[platform.value][maneuver.value] = (kr, mm_rl, mm_apn)

            print(f"{platform.value:<12} | {maneuver.value:<22} | {kr:>13.1f}% | {mm_rl:>16.2f} | {mm_apn:>17.2f}")
        print("-" * 92)

    return results


eval_results = evaluate_matrix(trained_models, episodes_per_cell=5)


In [ ]:
def plot_3d_intercept_showcase(models: Dict[PlatformType, PPO], maneuver: ManeuverType = ManeuverType.BARREL_ROLL):
    fig = plt.figure(figsize=(18, 5.5))
    for idx, platform in enumerate([PlatformType.QUADCOPTER, PlatformType.FIXED_WING, PlatformType.MISSILE], 1):
        env = InterceptorPathPlanningEnv(platform=platform, use_apn=True, fixed_maneuver=maneuver, seed=2026)
        obs, _ = env.reset()
        done = False
        while not done:
            action, _ = models[platform].predict(obs, deterministic=True)
            obs, _, terminated, truncated, info = env.step(action)
            done = terminated or truncated

        traj_M = np.array(env.traj_M)
        traj_T = np.array(env.traj_T)

        ax = fig.add_subplot(1, 3, idx, projection="3d")
        ax.plot(traj_T[:, 0], traj_T[:, 1], -traj_T[:, 2], "r--", lw=2, label="Evasive Target")
        ax.plot(traj_M[:, 0], traj_M[:, 1], -traj_M[:, 2], "b-", lw=2, label=f"{platform.value.title()} (APN_RL)")
        ax.scatter(traj_M[0, 0], traj_M[0, 1], -traj_M[0, 2], c="blue", s=50, marker="^", label="Interceptor Start")
        ax.scatter(traj_T[0, 0], traj_T[0, 1], -traj_T[0, 2], c="red", s=50, marker="o", label="Target Start")
        ax.scatter(traj_M[-1, 0], traj_M[-1, 1], -traj_M[-1, 2], c="green" if info["killed"] else "orange", s=90, marker="*", label="Intercept")

        ax.set_title(f"{platform.value.upper()} vs {maneuver.value}\nMiss: {info['miss_distance']:.2f} m | Killed: {info['killed']}")
        ax.set_xlabel("North X (m)")
        ax.set_ylabel("East Y (m)")
        ax.set_zlabel("Altitude -Z (m)")
        ax.legend(loc="best", fontsize=8)

    plt.tight_layout()
    plt.show()


plot_3d_intercept_showcase(trained_models, ManeuverType.BARREL_ROLL)
plot_3d_intercept_showcase(trained_models, ManeuverType.SPIRAL_CORKSCREW)
